# Find Common Point Between Two Sidewalk Images with LoFTR

Given two street-level images of (possibly overlapping) sidewalks, use **LoFTR** (via `kornia`)
to find dense feature correspondences. The highest-confidence match restricted to the
sidewalk region is reported as the **common point**.

Fetches images and sidewalk masks from GCS in the same way as `sidewalk_measurement.ipynb`.

## 1. Imports

In [ ]:
import os
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
import kornia as K
import kornia.feature as KF
from dotenv import load_dotenv
from google.cloud import storage

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('device:', device)

## 2. Configuration

Set the two image blob paths. Masks are auto-discovered under `v2/segmentation-results/{img_id}_{lat}_{lon}/{direction}/sidewalk/`.

In [ ]:
load_dotenv(os.path.join('..', '.env'))
GCS_BUCKET_NAME = os.environ.get('GCS_BUCKET_NAME', '')
GCP_PROJECT_ID  = os.environ.get('GCP_PROJECT_ID', '')

# Two images to match. Replace with your own pair.
IMAGE_NAME_A = "streetview/polygon_4v/20260404T133859Z/013-0274_forward_40.9722976_29.0668273_305.0.jpg"
IMAGE_NAME_B = "streetview/polygon_4v/20260404T133859Z/013-0273_backward_40.972349_29.0667298_125.0.jpg"

def masks_prefix_for(image_name: str) -> str:
    fn = os.path.basename(image_name)
    img_id, direction, lat, lon, _ = fn.replace('.jpg','').split('_')
    return f"v3/segmentation-results/{img_id[4:8]}_{lat}_{lon}/{direction}"

MASKS_PREFIX_A = masks_prefix_for(IMAGE_NAME_A)
MASKS_PREFIX_B = masks_prefix_for(IMAGE_NAME_B)
print('A masks prefix:', MASKS_PREFIX_A)
print('B masks prefix:', MASKS_PREFIX_B)

# LoFTR parameters
LOFTR_WEIGHTS = 'outdoor'       # 'outdoor' or 'indoor'
LOFTR_LONG_SIDE = 840           # resize long side to this before inference
MIN_CONFIDENCE = 0.5            # drop matches below this LoFTR confidence
RESTRICT_TO_SIDEWALK = True     # keep only matches where both ends fall inside sidewalk mask

## 3. GCS Loaders (image + combined sidewalk mask)

In [ ]:
def _gcs_bucket():
    return storage.Client(project=GCP_PROJECT_ID).bucket(GCS_BUCKET_NAME)

def download_image_from_gcs(blob_name: str) -> np.ndarray:
    data = _gcs_bucket().blob(blob_name).download_as_bytes()
    return np.array(Image.open(io.BytesIO(data)).convert('RGB'))

def download_mask_from_gcs(blob_name: str) -> np.ndarray:
    data = _gcs_bucket().blob(blob_name).download_as_bytes()
    m = np.array(Image.open(io.BytesIO(data)).convert('L'))
    return (m > 127).astype(bool)

def list_blobs_under(prefix: str):
    return [b.name for b in _gcs_bucket().list_blobs(prefix=prefix)]

def load_individual_sidewalk_masks(masks_prefix: str, shape):
    """Load all sidewalk masks and identify their side (left/right)."""
    blobs = [b for b in list_blobs_under(masks_prefix + '/sidewalk/') if b.endswith('.png')]
    masks = []
    W = shape[1]
    
    for b in blobs:
        m = download_mask_from_gcs(b)
        if m.shape != shape:
            m = cv2.resize(m.astype(np.uint8), (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
        
        # Determine side based on center of mass
        cols = np.where(m.any(axis=0))[0]
        if len(cols) == 0:
            continue
            
        center_x = np.mean(cols)
        side = 'left' if center_x < W / 2 else 'right'
        
        masks.append({
            'mask': m,
            'side': side,
            'blob_name': b
        })
        
    return masks

print('Fetching image A...')
img_A = download_image_from_gcs(IMAGE_NAME_A)
print(f'  {img_A.shape}')
print('Fetching image B...')
img_B = download_image_from_gcs(IMAGE_NAME_B)
print(f'  {img_B.shape}')

print('Fetching sidewalk masks A...')
masks_A_list = load_individual_sidewalk_masks(MASKS_PREFIX_A, img_A.shape[:2])
print(f'  {len(masks_A_list)} valid mask(s)')
for i, m in enumerate(masks_A_list):
    print(f"    Mask {i}: Side={m['side']}, Area={m['mask'].sum():,} px")

print('Fetching sidewalk masks B...')
masks_B_list = load_individual_sidewalk_masks(MASKS_PREFIX_B, img_B.shape[:2])
print(f'  {len(masks_B_list)} valid mask(s)')
for i, m in enumerate(masks_B_list):
    print(f"    Mask {i}: Side={m['side']}, Area={m['mask'].sum():,} px")

# Visualize masks
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_A)
for m in masks_A_list:
    axes[0].imshow(m['mask'], alpha=0.4, cmap='Reds' if m['side'] == 'left' else 'Greens')
axes[0].set_title('Image A - Left(Red)/Right(Green)')
axes[0].axis('off')

axes[1].imshow(img_B)
for m in masks_B_list:
    axes[1].imshow(m['mask'], alpha=0.4, cmap='Reds' if m['side'] == 'left' else 'Greens')
axes[1].set_title('Image B - Left(Red)/Right(Green)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Run LoFTR

In [ ]:
def to_loftr_tensor(img_rgb: np.ndarray, long_side: int):
    """Return (gray tensor [1,1,H',W'] in [0,1], scale_x, scale_y) where scale maps resized->original."""
    H, W = img_rgb.shape[:2]
    s = long_side / max(H, W)
    newW, newH = int(round(W * s)), int(round(H * s))
    # LoFTR expects dims divisible by 8
    newW -= newW % 8
    newH -= newH % 8
    
    # Ensure minimum size to prevent cv2.resize crash (inv_scale_x > 0 error)
    newW = max(8, newW)
    newH = max(8, newH)
    
    img_resized = cv2.resize(img_rgb, (newW, newH), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    t = torch.from_numpy(gray).float()[None, None] / 255.0
    scale_x = W / newW
    scale_y = H / newH
    return t, scale_x, scale_y

print('Loading LoFTR weights:', LOFTR_WEIGHTS)
matcher = KF.LoFTR(pretrained=LOFTR_WEIGHTS).eval().to(device)

def stack_side_by_side(a, b):
    h = max(a.shape[0], b.shape[0])
    pad = lambda x: np.pad(x, ((0, h - x.shape[0]), (0, 0), (0, 0)))
    return np.concatenate([pad(a), pad(b)], axis=1), a.shape[1]

## 8. Rectify Images
Using the geometry-based perspective rectification algorithm from `sidewalk_measurement.ipynb`.

In [ ]:
from sklearn.linear_model import RANSACRegressor, LinearRegression

BORDER_MARGIN = 3
RANSAC_RESIDUAL_THRESHOLD = 1.0  # px — edges further than this from consensus are outliers
RANSAC_MIN_SAMPLES = 0.3         # at least half the rows must agree
SHIFT_PERCENTILE = 10             # shift fitted line to 5th percentile of inliers for outer envelope
ROAD_TOUCH_MARGIN = 20            # px — sidewalk edge within this distance of road = confirmed edge
ROAD_CONFIRMED_WEIGHT = 5.0       # weight multiplier for road-confirmed rows in fitting

def find_row_edges(mask: np.ndarray, road_mask: np.ndarray = None, border_margin: int = BORDER_MARGIN,
                   ransac_threshold: float = RANSAC_RESIDUAL_THRESHOLD,
                   ransac_min_samples: float = RANSAC_MIN_SAMPLES,
                   road_touch_margin: int = ROAD_TOUCH_MARGIN):
    """
    For each row, find left and right sidewalk edges.
    
    Road-aware edge confirmation:
      If a sidewalk edge is within road_touch_margin of a road pixel,
      that edge is "road-confirmed" — guaranteed correct.
      Road-confirmed rows are forced inliers and given high weight
      in line fitting, since they represent the true sidewalk boundary.
    
    Left and right edges are validated independently.
    Uses RANSAC + directional inlier/outlier reclassification:
    - Left edge: points to the LEFT of fit are always inliers (can't be occlusion),
                 points to the RIGHT beyond threshold are always outliers (occlusion).
    - Right edge: mirrored logic.
    After reclassification, refits using only clean points.

    Returns (left_edges, right_edges, valid_mask, extrapolated_mask, edge_model).
    """
    # ── Determine which side of the road the sidewalk is on ──
    sw_center = np.mean(np.where(mask.any(axis=0))[0])
    if road_mask is not None and np.any(road_mask):
        road_center = np.mean(np.where(road_mask.any(axis=0))[0])
        is_left_sidewalk = sw_center < road_center
    else:
        is_left_sidewalk = None

    H, W = mask.shape
    left_edges = np.full(H, np.nan)
    right_edges = np.full(H, np.nan)
    valid = np.zeros(H, dtype=bool)
    extrapolated = np.zeros(H, dtype=bool)

    left_valid = np.zeros(H, dtype=bool)
    right_valid = np.zeros(H, dtype=bool)
    left_clipped = np.zeros(H, dtype=bool)
    right_clipped = np.zeros(H, dtype=bool)
    has_sidewalk = np.zeros(H, dtype=bool)

    # ── Road-confirmed tracking ──
    road_confirmed_L = np.zeros(H, dtype=bool)
    road_confirmed_R = np.zeros(H, dtype=bool)

    for r in range(H):
        cols = np.where(mask[r])[0]
        if len(cols) == 0:
            continue
        has_sidewalk[r] = True
        left_col = cols[0]
        right_col = cols[-1]

        left_edges[r] = left_col
        right_edges[r] = right_col

        left_at_border = left_col < border_margin
        right_at_border = right_col >= W - border_margin

        if not left_at_border:
            left_valid[r] = True
        else:
            left_clipped[r] = True

        if not right_at_border:
            right_valid[r] = True
        else:
            right_clipped[r] = True

        # ── Road proximity confirmation ──
        if road_mask is not None and is_left_sidewalk is not None:
            road_cols = np.where(road_mask[r])[0]
            if len(road_cols) > 0:
                if is_left_sidewalk:
                    # Sidewalk is LEFT of road → RIGHT edge is road-side
                    dist_right = np.min(np.abs(road_cols - right_col))
                    if dist_right <= road_touch_margin:
                        road_confirmed_R[r] = True
                        right_valid[r] = True   # override border-clip
                        right_clipped[r] = False
                    else:
                        # Road exists in this row but doesn't touch → suspect edge
                        right_valid[r] = False
                else:
                    # Sidewalk is RIGHT of road → LEFT edge is road-side
                    dist_left = np.min(np.abs(road_cols - left_col))
                    if dist_left <= road_touch_margin:
                        road_confirmed_L[r] = True
                        left_valid[r] = True    # override border-clip
                        left_clipped[r] = False
                    else:
                        left_valid[r] = False

    both_valid = left_valid & right_valid
    left_valid_idx = np.where(left_valid)[0]
    right_valid_idx = np.where(right_valid)[0]

    n_road_confirmed_L = road_confirmed_L[left_valid_idx].sum() if len(left_valid_idx) > 0 else 0
    n_road_confirmed_R = road_confirmed_R[right_valid_idx].sum() if len(right_valid_idx) > 0 else 0
    print(f"    Road-confirmed edges — left: {n_road_confirmed_L}, right: {n_road_confirmed_R}")

    edge_model = None

    if len(left_valid_idx) >= 2 and len(right_valid_idx) >= 2:

        # ══════════════════════════════════════════════════════
        # LEFT EDGE
        # ══════════════════════════════════════════════════════
        a_L, b_L, inlier_mask_L = _fit_edge(
            left_valid_idx, left_edges, road_confirmed_L,
            side="left",
            ransac_threshold=ransac_threshold,
            ransac_min_samples=ransac_min_samples,
        )

        # ══════════════════════════════════════════════════════
        # RIGHT EDGE
        # ══════════════════════════════════════════════════════
        a_R, b_R, inlier_mask_R = _fit_edge(
            right_valid_idx, right_edges, road_confirmed_R,
            side="right",
            ransac_threshold=ransac_threshold,
            ransac_min_samples=ransac_min_samples,
        )

        edge_model = {
            'a_L': a_L, 'b_L': b_L,
            'a_R': a_R, 'b_R': b_R,
            'inlier_mask_L': inlier_mask_L,
            'inlier_mask_R': inlier_mask_R,
            'left_valid_idx': left_valid_idx,
            'right_valid_idx': right_valid_idx,
            'valid_idx': np.where(both_valid)[0],
            'road_confirmed_L': road_confirmed_L,
            'road_confirmed_R': road_confirmed_R,
        }

        # Extrapolate clipped edges from the model
        for r in np.where(has_sidewalk)[0]:
            if left_clipped[r]:
                left_edges[r] = a_L * r + b_L
            if right_clipped[r]:
                right_edges[r] = a_R * r + b_R
            if left_clipped[r] or right_clipped[r]:
                extrapolated[r] = True

        valid = has_sidewalk.copy()

    else:
        valid = both_valid.copy()

    return left_edges, right_edges, valid, extrapolated, edge_model


def _fit_edge(valid_idx, edges, road_confirmed, side="left",
              ransac_threshold=RANSAC_RESIDUAL_THRESHOLD,
              ransac_min_samples=RANSAC_MIN_SAMPLES):
    """
    Fit a single edge (left or right) with road-confirmed priority.
    
    If enough road-confirmed rows exist (>= 4), fit primarily from those
    since they are guaranteed correct. Road-confirmed rows are always
    forced as inliers.
    
    side="left":  outliers are points pushed RIGHT  (residual > threshold)
    side="right": outliers are points pushed LEFT   (residual < -threshold)
    """
    rc_mask = road_confirmed[valid_idx]  # which valid rows are road-confirmed
    n_rc = rc_mask.sum()
    rc_rows = valid_idx[rc_mask]
    
    is_left = (side == "left")

    if n_rc >= 4:
        # ── Road-confirmed priority fit ──
        # Fit line from confirmed rows only (ground truth)
        a, b = np.polyfit(rc_rows.astype(float), edges[rc_rows], 1)
        
        # Envelope shift: use road-confirmed rows as reference
        fitted_rc = a * rc_rows.astype(float) + b
        residuals_rc = edges[rc_rows] - fitted_rc
        if is_left:
            shift = np.percentile(residuals_rc, SHIFT_PERCENTILE)
        else:
            shift = np.percentile(residuals_rc, 100 - SHIFT_PERCENTILE)
        b += shift

        # Classify all valid rows against this road-anchored line
        fitted_all = a * valid_idx.astype(float) + b
        residuals_all = edges[valid_idx] - fitted_all
        if is_left:
            inlier_mask = (residuals_all < ransac_threshold)
        else:
            inlier_mask = (residuals_all > -ransac_threshold)
        
        # Force road-confirmed rows as inliers (they are ground truth)
        inlier_mask[rc_mask] = True

        # Optional refit with all inliers (road-confirmed get high weight)
        if inlier_mask.sum() >= 2:
            clean_idx = valid_idx[inlier_mask]
            weights = np.ones(len(clean_idx), dtype=float)
            for i, r in enumerate(clean_idx):
                if road_confirmed[r]:
                    weights[i] = ROAD_CONFIRMED_WEIGHT
            # Weighted polyfit
            a, b = np.polyfit(clean_idx.astype(float), edges[clean_idx], 1, w=weights)
            
            # Re-apply envelope shift
            fitted2 = a * rc_rows.astype(float) + b
            residuals2 = edges[rc_rows] - fitted2
            if is_left:
                shift = np.percentile(residuals2, SHIFT_PERCENTILE)
            else:
                shift = np.percentile(residuals2, 100 - SHIFT_PERCENTILE)
            b += shift
            
            # Final inlier classification
            fitted3 = a * valid_idx.astype(float) + b
            residuals3 = edges[valid_idx] - fitted3
            if is_left:
                inlier_mask = (residuals3 < ransac_threshold)
            else:
                inlier_mask = (residuals3 > -ransac_threshold)
            inlier_mask[rc_mask] = True
        
        print(f"    {side.capitalize()} edge: road-confirmed fit ({n_rc} anchor rows, "
              f"{inlier_mask.sum()} total inliers)")
        return a, b, inlier_mask

    # ── Fallback: standard RANSAC (no/insufficient road confirmation) ──
    if len(valid_idx) >= 4:
        X = valid_idx.reshape(-1, 1).astype(float)
        
        # If we have SOME road-confirmed rows, use them as sample weight hints
        sample_weights = np.ones(len(valid_idx), dtype=float)
        if n_rc > 0:
            sample_weights[rc_mask] = ROAD_CONFIRMED_WEIGHT
        
        ransac = RANSACRegressor(
            estimator=LinearRegression(),
            residual_threshold=ransac_threshold,
            min_samples=ransac_min_samples,
            random_state=42
        )
        ransac.fit(X, edges[valid_idx], sample_weight=sample_weights)
        a = ransac.estimator_.coef_[0]
        b = ransac.estimator_.intercept_

        # Directional reclassification
        fitted = a * valid_idx.astype(float) + b
        residuals = edges[valid_idx] - fitted

        if is_left:
            inlier_mask = (residuals < ransac_threshold)
        else:
            inlier_mask = (residuals > -ransac_threshold)

        # Force road-confirmed as inliers
        if n_rc > 0:
            inlier_mask[rc_mask] = True

        if inlier_mask.sum() >= 2:
            clean_idx = valid_idx[inlier_mask]
            weights = np.ones(len(clean_idx), dtype=float)
            for i, r in enumerate(clean_idx):
                if road_confirmed[r]:
                    weights[i] = ROAD_CONFIRMED_WEIGHT
            a, b = np.polyfit(clean_idx.astype(float), edges[clean_idx], 1, w=weights)

            # Envelope shift
            fitted2 = a * clean_idx.astype(float) + b
            residuals_clean = edges[clean_idx] - fitted2
            if is_left:
                shift = np.percentile(residuals_clean, SHIFT_PERCENTILE)
            else:
                shift = np.percentile(residuals_clean, 100 - SHIFT_PERCENTILE)
            b += shift

            # Recompute inlier mask after shift
            fitted3 = a * valid_idx.astype(float) + b
            residuals3 = edges[valid_idx] - fitted3
            if is_left:
                inlier_mask = (residuals3 < ransac_threshold)
            else:
                inlier_mask = (residuals3 > -ransac_threshold)
            if n_rc > 0:
                inlier_mask[rc_mask] = True
    else:
        a, b = np.polyfit(valid_idx.astype(float), edges[valid_idx], 1)
        inlier_mask = np.ones(len(valid_idx), dtype=bool)

    return a, b, inlier_mask


def rectify_sidewalk(image_or_mask, left_edges, right_edges, valid_rows,
                     edge_model=None, target_width=None, is_mask=False,
                     cos_correction=1.0, f_px=None):
    """
    Per-row warp with horizontal AND vertical perspective correction.
    Samples along ground-plane perpendicular cross-sections using the
    perpendicular vanishing point for geometrically correct rectification.
    """
    H, W = image_or_mask.shape[:2]
    cy = H / 2.0

    valid_widths = right_edges[valid_rows] - left_edges[valid_rows]
    if len(valid_widths) == 0:
        return image_or_mask, target_width or 100, 0
    if target_width is None:
        target_width = int(np.median(valid_widths) * cos_correction)

    valid_idx = np.where(valid_rows)[0]
    all_rows = np.arange(H)

    left_interp = np.full(H, np.nan)
    right_interp = np.full(H, np.nan)

    if len(valid_idx) < 2:
        left_interp[:] = (left_edges[valid_idx[0]] if len(valid_idx) else 0)
        right_interp[:] = (right_edges[valid_idx[0]] if len(valid_idx) else W - 1)
        vy = -1e8
        first_valid = 0
        last_valid = H - 1
        a_L = 0.0
        b_L = left_interp[0]
        a_R = 0.0
        b_R = right_interp[0]
    else:
        if edge_model is not None:
            a_L, b_L = edge_model['a_L'], edge_model['b_L']
            a_R, b_R = edge_model['a_R'], edge_model['b_R']
        else:
            valid_r = valid_idx.astype(float)
            a_L, b_L = np.polyfit(valid_r, left_edges[valid_idx], 1)
            a_R, b_R = np.polyfit(valid_r, right_edges[valid_idx], 1)

        left_interp = a_L * all_rows + b_L
        right_interp = a_R * all_rows + b_R

        if abs(a_L - a_R) > 1e-6:
            vy = (b_R - b_L) / (a_L - a_R)
        else:
            vy = -1e8

        first_valid = valid_idx[0]
        last_valid = valid_idx[-1]

    # ── Perpendicular vanishing point ──
    use_perp = False
    vp_perp_x = 0.0
    if f_px is not None and abs(a_L - a_R) > 1e-6:
        vp_x = a_L * vy + b_L
        vp_x_offset = vp_x - W / 2.0
        
        if abs(vp_x_offset) > 1e-6:
            a_avg = (a_L + a_R) / 2.0
            sign = 1.0 if a_avg > 0 else -1.0
            vp_perp_x = W / 2.0 + sign * f_px * f_px / abs(vp_x_offset)
            use_perp = True

    # ── Vertical perspective correction ──
    row_scale = np.ones(H, dtype=np.float64)
    ref_dist = abs(last_valid - vy)
    MAX_STRETCH = 50.0

    for r in range(first_valid, last_valid + 1):
        dist = abs(r - vy)
        if dist > 0:
            s = (ref_dist / dist) ** 2
        else:
            s = MAX_STRETCH
        row_scale[r] = min(s, MAX_STRETCH)

    cum_real = np.cumsum(row_scale)
    cum_real = cum_real - cum_real[0]
    total_real_height = cum_real[-1]

    out_height = int(np.ceil(total_real_height)) + 1
    out_rows = np.arange(out_height, dtype=np.float32)
    src_row_for_out = np.interp(out_rows, cum_real, np.arange(H, dtype=np.float32))

    padding = int(target_width * 0.3)
    out_width = target_width + 2 * padding

    map_x = np.zeros((out_height, out_width), dtype=np.float32)
    map_y = np.zeros((out_height, out_width), dtype=np.float32)

    out_cols = np.arange(out_width, dtype=np.float32)

    for out_r in range(out_height):
        src_r = src_row_for_out[out_r]
        src_r_int = int(src_r)

        L = left_interp[min(src_r_int, H - 1)]
        R = right_interp[min(src_r_int, H - 1)]

        src_r_frac = src_r - src_r_int
        if src_r_frac > 0 and src_r_int + 1 < H:
            L_next = left_interp[src_r_int + 1]
            R_next = right_interp[src_r_int + 1]
            L = L * (1 - src_r_frac) + L_next * src_r_frac
            R = R * (1 - src_r_frac) + R_next * src_r_frac

        src_width = R - L
        if src_width <= 0:
            map_x[out_r, :] = -1
            map_y[out_r, :] = src_r
            continue

        cx = (L + R) / 2.0

        if use_perp and src_r > cy + 1:
            dx_perp = vp_perp_x - cx
            if abs(dx_perp) > 1e-6:
                perp_slope = (cy - src_r) / dx_perp

                denom_L = 1.0 - a_L * perp_slope
                denom_R = 1.0 - a_R * perp_slope

                if abs(denom_L) > 1e-9 and abs(denom_R) > 1e-9:
                    t_L = (L - cx) / denom_L
                    t_R = (R - cx) / denom_R
                    perp_span = t_R - t_L

                    if perp_span > 0:
                        t = t_L + (out_cols - padding) / target_width * perp_span
                        map_x[out_r, :] = cx + t
                        map_y[out_r, :] = src_r + t * perp_slope
                        continue

        # Fallback: horizontal sampling
        scale = src_width / target_width
        src_cols = L + (out_cols - padding) * scale
        map_x[out_r, :] = src_cols
        map_y[out_r, :] = src_r

    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    inp = image_or_mask.astype(np.uint8) if is_mask else image_or_mask
    warped = cv2.remap(inp, map_x, map_y, interpolation=flags, borderMode=cv2.BORDER_CONSTANT)

    if is_mask:
        warped = warped.astype(bool)

    return warped, target_width, padding

## 8a. Debugging Visualization: Rectification Lines 
Let's iterate over each mask from Image A and Image B and show the lines `find_row_edges` finds, along with the rectified image, to verify our inputs before-and-after geometry.

In [ ]:
def visualize_rectification(img, mask_item, title_prefix="Image"):
    single_mask = mask_item['mask']
    side = mask_item['side']
    
    left, right, valid, extrap, model = find_row_edges(single_mask)
    f_px = img.shape[1] / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
    
    if model is not None:
        vp_y = (model['b_R'] - model['b_L']) / (model['a_L'] - model['a_R']) if abs(model['a_L'] - model['a_R']) > 1e-6 else 0
        vp_x = model['a_L'] * vp_y + model['b_L']
        cos_corr = np.cos(max(0, np.arctan((vp_x - img.shape[1] / 2.0) / f_px)))
    else:
        cos_corr = 1.0
        
    rectified_img, w, pad = rectify_sidewalk(img, left, right, valid, edge_model=model, f_px=f_px, cos_correction=cos_corr)
    rectified_mask, _, _ = rectify_sidewalk(single_mask, left, right, valid, edge_model=model, target_width=w, is_mask=True, f_px=f_px, cos_correction=cos_corr)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 1. Show Original image + edges
    axes[0].imshow(img)
    axes[0].imshow(single_mask, cmap='Blues', alpha=0.3)
    
    H = single_mask.shape[0]
    valid_idx = np.where(valid)[0]
    if len(valid_idx) > 0:
        a_L, b_L = model['a_L'], model['b_L'] if model else (0,0)
        a_R, b_R = model['a_R'], model['b_R'] if model else (0,img.shape[1])
        
        y_vals = np.arange(H)
        x_left = a_L * y_vals + b_L
        x_right = a_R * y_vals + b_R
        
        # Plot model predictions
        axes[0].plot(x_left, y_vals, color='red', linewidth=2, label="Modeled Left")
        axes[0].plot(x_right, y_vals, color='lime', linewidth=2, label="Modeled Right")
        
    axes[0].set_xlim([0, img.shape[1]])
    axes[0].set_ylim([img.shape[0], 0])
    axes[0].legend(loc="lower right")
    axes[0].set_title(f"{title_prefix} - {side.capitalize()} Sidewalk Mask")
    axes[0].axis('off')
    
    # 2. Rectified Image
    axes[1].imshow(rectified_img)
    axes[1].set_title(f"Rectified Image")
    axes[1].axis('off')
    
    # 3. Rectified Mask
    axes[2].imshow(rectified_mask, cmap='Blues')
    axes[2].set_title(f"Rectified Mask")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("== Debugging Image A Masks ==")
for i, mA in enumerate(masks_A_list):
    print(f"\nMask A_{i} ({mA['side']})")
    visualize_rectification(img_A, mA, title_prefix=f"Image A [{i}]")
    
print("\n== Debugging Image B Masks ==")
for i, mB in enumerate(masks_B_list):
    print(f"\nMask B_{i} ({mB['side']})")
    visualize_rectification(img_B, mB, title_prefix=f"Image B [{i}]")

## 8. Process Valid Pairs and Run LoFTR

For each valid pair (A: left vs B: right OR A: right vs B: left):
1. Compute edges and rectify images based on the specific masks.
2. Crop the top 40% of the rectified images (further parts that are heavily distorted).
3. Flip the cropped rectified image B vertically and horizontally (`cv2.flip(img, -1)`).
4. Run LoFTR matching on these processed crops.
5. If matches are found, display them.

In [ ]:
# Keep track of the best match across all pairs
global_best_conf = -1.0
global_best_data = None

for idx_A, mA in enumerate(masks_A_list):
    for idx_B, mB in enumerate(masks_B_list):
        # We only match Left with Right, or Right with Left
        if mA['side'] == mB['side']:
            continue
            
        print(f"\n{'='*50}")
        print(f"Testing Pair: A (Mask {idx_A}, {mA['side']}) vs B (Mask {idx_B}, {mB['side']})")
        print(f"{'='*50}")
        
        mask_A_single = mA['mask']
        mask_B_single = mB['mask']
        
        # --- Rectify A ---
        left_A, right_A, valid_A, extrap_A, model_A = find_row_edges(mask_A_single)
        f_px_A = img_A.shape[1] / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
        if model_A is not None:
            vp_y = (model_A['b_R'] - model_A['b_L']) / (model_A['a_L'] - model_A['a_R']) if abs(model_A['a_L'] - model_A['a_R']) > 1e-6 else 0
            vp_x = model_A['a_L'] * vp_y + model_A['b_L']
            cos_corr_A = np.cos(max(0, np.arctan((vp_x - img_A.shape[1] / 2.0) / f_px_A)))
        else:
            cos_corr_A = 1.0
            
        rectified_A, w_A, pad_A = rectify_sidewalk(img_A, left_A, right_A, valid_A, edge_model=model_A, f_px=f_px_A, cos_correction=cos_corr_A)
        rectified_mask_A, _, _ = rectify_sidewalk(mask_A_single, left_A, right_A, valid_A, edge_model=model_A, target_width=w_A, is_mask=True, f_px=f_px_A, cos_correction=cos_corr_A)

        # --- Rectify B ---
        left_B, right_B, valid_B, extrap_B, model_B = find_row_edges(mask_B_single)
        f_px_B = img_B.shape[1] / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
        if model_B is not None:
            vp_y = (model_B['b_R'] - model_B['b_L']) / (model_B['a_L'] - model_B['a_R']) if abs(model_B['a_L'] - model_B['a_R']) > 1e-6 else 0
            vp_x = model_B['a_L'] * vp_y + model_B['b_L']
            cos_corr_B = np.cos(max(0, np.arctan((vp_x - img_B.shape[1] / 2.0) / f_px_B)))
        else:
            cos_corr_B = 1.0
            
        rectified_B, w_B, pad_B = rectify_sidewalk(img_B, left_B, right_B, valid_B, edge_model=model_B, f_px=f_px_B, cos_correction=cos_corr_B)
        rectified_mask_B, _, _ = rectify_sidewalk(mask_B_single, left_B, right_B, valid_B, edge_model=model_B, target_width=w_B, is_mask=True, f_px=f_px_B, cos_correction=cos_corr_B)

        if rectified_A.shape[0] < 10 or rectified_B.shape[0] < 10:
            print("Rectification failed or produced too small image. Skipping pair.")
            continue

        # --- Preparation for LoFTR (Crop & Flip) ---
        crop_h_A = int(rectified_A.shape[0] * 0.4)
        crop_A = rectified_A[crop_h_A:, :]

        crop_h_B = int(rectified_B.shape[0] * 0.4)
        crop_B = rectified_B[crop_h_B:, :]

        crop_mask_A = rectified_mask_A[crop_h_A:, :]
        crop_mask_B = rectified_mask_B[crop_h_B:, :]

        # Flip B
        flip_crop_B = cv2.flip(crop_B, -1)
        flip_crop_mask_B = cv2.flip(crop_mask_B.astype(np.uint8), -1).astype(bool)

        tA_r, sxA_r, syA_r = to_loftr_tensor(crop_A, LOFTR_LONG_SIDE)
        tB_r, sxB_r, syB_r = to_loftr_tensor(flip_crop_B, LOFTR_LONG_SIDE)

        with torch.inference_mode():
            out_r = matcher({'image0': tA_r.to(device), 'image1': tB_r.to(device)})

        kpts_A_r = out_r['keypoints0'].cpu().numpy() * np.array([sxA_r, syA_r])
        kpts_B_r = out_r['keypoints1'].cpu().numpy() * np.array([sxB_r, syB_r])
        conf_r   = out_r['confidence'].cpu().numpy()
        
        # --- Filter Matches ---
        keep_r = conf_r >= MIN_CONFIDENCE
        
        if RESTRICT_TO_SIDEWALK:
            def in_mask_rect(pts, mask):
                xs = np.clip(pts[:, 0].round().astype(int), 0, mask.shape[1] - 1)
                ys = np.clip(pts[:, 1].round().astype(int), 0, mask.shape[0] - 1)
                return mask[ys, xs]
                
            on_sw_r = in_mask_rect(kpts_A_r, crop_mask_A) & in_mask_rect(kpts_B_r, flip_crop_mask_B)
            keep_r &= on_sw_r

        pA_r = kpts_A_r[keep_r]
        pB_r = kpts_B_r[keep_r]
        pC_r = conf_r[keep_r]

        # RANSAC filtering
        if len(pA_r) >= 4:
            try:
                H_mat, inl_r = cv2.findHomography(pA_r, pB_r, cv2.RANSAC, 3.0)
                if inl_r is not None:
                    inl_r = inl_r.ravel().astype(bool)
                    pA_r, pB_r, pC_r = pA_r[inl_r], pB_r[inl_r], pC_r[inl_r]
            except cv2.error as e:
                pA_r, pB_r, pC_r = [], [], []

        print(f"Final valid matches for this pair: {len(pA_r)}")
        
        # ALWAYS plot the pair so we can see what they look like, even if 0 matches
        canvas_r, offset_r = stack_side_by_side(crop_A, flip_crop_B)
        fig_r, ax_r = plt.subplots(figsize=(12, 6))
        ax_r.imshow(canvas_r)

        if len(pA_r) > 0:
            best_idx = int(np.argmax(pC_r))
            best_conf = pC_r[best_idx]
            
            if best_conf > global_best_conf:
                global_best_conf = best_conf
                global_best_data = {
                    'A_idx': idx_A, 'B_idx': idx_B,
                    'A_side': mA['side'], 'B_side': mB['side'],
                    'pA_r': pA_r, 'pB_r': pB_r, 'pC_r': pC_r,
                    'best_idx': best_idx,
                    'crop_A': crop_A, 'flip_crop_B': flip_crop_B,
                    'rectified_A': rectified_A, 'rectified_B': rectified_B,
                    'rect_mask_A': rectified_mask_A, 'rect_mask_B': rectified_mask_B
                }
            
            for (xa, ya), (xb, yb), c in zip(pA_r, pB_r, pC_r):
                ax_r.plot([xa, xb + offset_r], [ya, yb], '-', color='lime', linewidth=0.5, alpha=0.4)
            
            xa, ya = pA_r[best_idx]
            xb, yb = pB_r[best_idx]
            ax_r.plot([xa, xb + offset_r], [ya, yb], '-', color='red', linewidth=2.0)
            ax_r.scatter([xa, xb + offset_r], [ya, yb], s=120, c='red', edgecolor='white', zorder=5)
            ax_r.set_title(f"A({mA['side']}) vs B({mB['side']}) Matches - Best Conf={best_conf:.3f}")
        else:
            ax_r.set_title(f"A({mA['side']}) vs B({mB['side']}) Matches - NO MATCHES FOUND")

        ax_r.axis('off')
        plt.tight_layout()
        plt.show()

# Print global best result summary
if global_best_data is not None:
    best_conf = global_best_conf
    A_side, B_side = global_best_data['A_side'], global_best_data['B_side']
    print(f"\n>>> BEST OVERALL MATCH: A (Mask {global_best_data['A_idx']}, {A_side}) vs B (Mask {global_best_data['B_idx']}, {B_side}) with confidence {best_conf:.3f} <<<\n")
else:
    print("\n>>> NO MATCHES FOUND ACROSS ANY PAIRS <<<")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

def get_manual_edges(img_disp, title):
    points = []
    clean_copy = img_disp.copy()
    
    def mouse_callback(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            points.append((x, y))
            cv2.circle(img_disp, (x, y), 5, (0, 255, 0), -1)
            # Draw lines if we have pairs
            if len(points) == 2: # Left line
                cv2.line(img_disp, points[0], points[1], (0, 255, 0), 2)
            elif len(points) == 4: # Right line
                cv2.line(img_disp, points[2], points[3], (0, 255, 0), 2)
            cv2.imshow(title, img_disp)
            
    cv2.imshow(title, img_disp)
    cv2.setMouseCallback(title, mouse_callback)
    
    print(f"\n--- {title} ---")
    print("  -> Click 4 points: 1) Top-Left, 2) Bottom-Left, 3) Top-Right, 4) Bottom-Right")
    print("  -> Press 'c' to clear points and restart.")
    print("  -> Press 'q' to skip this mask.")
    
    while True:
        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'):
            cv2.destroyWindow(title)
            cv2.waitKey(1)
            return None
        elif key == ord('c'):
            points.clear()
            img_disp[:] = clean_copy[:]
            cv2.imshow(title, img_disp)
            print("  -> Cleared. Click 4 points again.")
        
        if len(points) == 4:
            cv2.waitKey(500)
            break
            
    cv2.destroyWindow(title)
    cv2.waitKey(1) # macOS glitch fix
    return points

def manual_find_row_edges(mask, img, title="Image"):
    # Blend image and mask for display
    mask_disp = cv2.cvtColor((mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    mask_disp[:,:,0] = 0 # Make mask green instead of white
    mask_disp[:,:,2] = 0
    disp = cv2.addWeighted(img, 0.7, mask_disp, 0.3, 0)
    
    pts = get_manual_edges(disp, title)
    if pts is None:
        return None, None, None, None, None
        
    tl, bl, tr, br = pts
    
    # line: x = a*y + b
    if bl[1] == tl[1]: bl = (bl[0], bl[1]+1) # prevent div by zero
    a_L = (bl[0] - tl[0]) / (bl[1] - tl[1])
    b_L = tl[0] - a_L * tl[1]
    
    if br[1] == tr[1]: br = (br[0], br[1]+1)
    a_R = (br[0] - tr[0]) / (br[1] - tr[1])
    b_R = tr[0] - a_R * tr[1]
    
    model = {
        'a_L': a_L, 'b_L': b_L, 'c_L': None, 'd_L': None,
        'a_R': a_R, 'b_R': b_R, 'c_R': None, 'd_R': None
    }
    
    H = mask.shape[0]
    y_vals = np.arange(H)
    left = a_L * y_vals + b_L
    right = a_R * y_vals + b_R
    # Use where the mask physically exists to bound the vertical perspective stretching
    valid = np.any(mask > 0, axis=1)
    extrap = np.zeros(H, dtype=bool)
    
    return left, right, valid, extrap, model

# Gather Manual Models for A
print("\n[STEP 1] Manuel Edge Selection for Image A")
manual_models_A = []
for idx, mA in enumerate(masks_A_list):
    l, r, v, e, m = manual_find_row_edges(mA['mask'], img_A, f"Image A - Mask {idx} ({mA['side']})")
    manual_models_A.append((l, r, v, e, m))

# Gather Manual Models for B
print("\n[STEP 2] Manuel Edge Selection for Image B")
manual_models_B = []
for idx, mB in enumerate(masks_B_list):
    l, r, v, e, m = manual_find_row_edges(mB['mask'], img_B, f"Image B - Mask {idx} ({mB['side']})")
    manual_models_B.append((l, r, v, e, m))


print("\n[STEP 3] Running the rest of the matches using manual points...")
global_best_conf = -1.0
global_best_data = None

for idx_A, mA in enumerate(masks_A_list):
    for idx_B, mB in enumerate(masks_B_list):
        if mA['side'] == mB['side']:
            continue
            
        print(f"\n{'='*50}")
        print(f"Testing Pair: A (Mask {idx_A}, {mA['side']}) vs B (Mask {idx_B}, {mB['side']})")
        print(f"{'='*50}")
        
        left_A, right_A, valid_A, extrap_A, model_A = manual_models_A[idx_A]
        left_B, right_B, valid_B, extrap_B, model_B = manual_models_B[idx_B]
        
        if model_A is None or model_B is None:
            print("Missing manual model for this pair (Skipped during selection).")
            continue
        
        mask_A_single = mA['mask']
        mask_B_single = mB['mask']
        
        # --- Rectify A ---
        f_px_A = img_A.shape[1] / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
        vp_y = (model_A['b_R'] - model_A['b_L']) / (model_A['a_L'] - model_A['a_R']) if abs(model_A['a_L'] - model_A['a_R']) > 1e-6 else 0
        vp_x = model_A['a_L'] * vp_y + model_A['b_L']
        cos_corr_A = np.cos(max(0, np.arctan((vp_x - img_A.shape[1] / 2.0) / f_px_A)))
            
        rectified_A, w_A, pad_A = rectify_sidewalk(img_A, left_A, right_A, valid_A, edge_model=model_A, f_px=f_px_A, cos_correction=cos_corr_A)
        rectified_mask_A, _, _ = rectify_sidewalk(mask_A_single, left_A, right_A, valid_A, edge_model=model_A, target_width=w_A, is_mask=True, f_px=f_px_A, cos_correction=cos_corr_A)

        # --- Rectify B ---
        f_px_B = img_B.shape[1] / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
        vp_y_B = (model_B['b_R'] - model_B['b_L']) / (model_B['a_L'] - model_B['a_R']) if abs(model_B['a_L'] - model_B['a_R']) > 1e-6 else 0
        vp_x_B = model_B['a_L'] * vp_y_B + model_B['b_L']
        cos_corr_B = np.cos(max(0, np.arctan((vp_x_B - img_B.shape[1] / 2.0) / f_px_B)))
            
        rectified_B, w_B, pad_B = rectify_sidewalk(img_B, left_B, right_B, valid_B, edge_model=model_B, f_px=f_px_B, cos_correction=cos_corr_B)
        rectified_mask_B, _, _ = rectify_sidewalk(mask_B_single, left_B, right_B, valid_B, edge_model=model_B, target_width=w_B, is_mask=True, f_px=f_px_B, cos_correction=cos_corr_B)

        if rectified_A.shape[0] < 10 or rectified_B.shape[0] < 10:
            print("Rectification failed or produced too small image. Skipping pair.")
            continue

        # --- Preparation for LoFTR (Crop & Flip) ---
        crop_h_A = int(rectified_A.shape[0] * 0.4)
        crop_A = rectified_A[crop_h_A:, :]

        crop_h_B = int(rectified_B.shape[0] * 0.4)
        crop_B = rectified_B[crop_h_B:, :]

        crop_mask_A = rectified_mask_A[crop_h_A:, :]
        crop_mask_B = rectified_mask_B[crop_h_B:, :]

        # Flip B
        flip_crop_B = cv2.flip(crop_B, -1)
        flip_crop_mask_B = cv2.flip(crop_mask_B.astype(np.uint8), -1).astype(bool)

        tA_r, sxA_r, syA_r = to_loftr_tensor(crop_A, LOFTR_LONG_SIDE)
        tB_r, sxB_r, syB_r = to_loftr_tensor(flip_crop_B, LOFTR_LONG_SIDE)

        with torch.inference_mode():
            out_r = matcher({'image0': tA_r.to(device), 'image1': tB_r.to(device)})

        kpts_A_r = out_r['keypoints0'].cpu().numpy() * np.array([sxA_r, syA_r])
        kpts_B_r = out_r['keypoints1'].cpu().numpy() * np.array([sxB_r, syB_r])
        conf_r   = out_r['confidence'].cpu().numpy()
        
        # --- Filter Matches ---
        keep_r = conf_r >= MIN_CONFIDENCE
        
        if RESTRICT_TO_SIDEWALK:
            def in_mask_rect(pts, mask):
                xs = np.clip(pts[:, 0].round().astype(int), 0, mask.shape[1] - 1)
                ys = np.clip(pts[:, 1].round().astype(int), 0, mask.shape[0] - 1)
                return mask[ys, xs]
                
            on_sw_r = in_mask_rect(kpts_A_r, crop_mask_A) & in_mask_rect(kpts_B_r, flip_crop_mask_B)
            keep_r &= on_sw_r

        pA_r = kpts_A_r[keep_r]
        pB_r = kpts_B_r[keep_r]
        pC_r = conf_r[keep_r]

        # RANSAC filtering
        if len(pA_r) >= 4:
            try:
                H_mat, inl_r = cv2.findHomography(pA_r, pB_r, cv2.RANSAC, 3.0)
                if inl_r is not None:
                    inl_r = inl_r.ravel().astype(bool)
                    pA_r, pB_r, pC_r = pA_r[inl_r], pB_r[inl_r], pC_r[inl_r]
            except cv2.error as e:
                pA_r, pB_r, pC_r = [], [], []

        print(f"Final valid matches for this pair: {len(pA_r)}")
        
        # ALWAYS plot the pair so we can see what they look like, even if 0 matches
        canvas_r, offset_r = stack_side_by_side(crop_A, flip_crop_B)
        fig_r, ax_r = plt.subplots(figsize=(12, 6))
        ax_r.imshow(canvas_r)

        if len(pA_r) > 0:
            best_idx = int(np.argmax(pC_r))
            best_conf = pC_r[best_idx]
            
            if best_conf > global_best_conf:
                global_best_conf = best_conf
                global_best_data = {
                    'A_idx': idx_A, 'B_idx': idx_B,
                    'A_side': mA['side'], 'B_side': mB['side'],
                    'pA_r': pA_r, 'pB_r': pB_r, 'pC_r': pC_r,
                    'best_idx': best_idx,
                    'crop_A': crop_A, 'flip_crop_B': flip_crop_B,
                    'rectified_A': rectified_A, 'rectified_B': rectified_B,
                    'rect_mask_A': rectified_mask_A, 'rect_mask_B': rectified_mask_B
                }
            
            for (xa, ya), (xb, yb), c in zip(pA_r, pB_r, pC_r):
                ax_r.plot([xa, xb + offset_r], [ya, yb], '-', color='lime', linewidth=0.5, alpha=0.4)
            
            xa, ya = pA_r[best_idx]
            xb, yb = pB_r[best_idx]
            ax_r.plot([xa, xb + offset_r], [ya, yb], '-', color='red', linewidth=2.0)
            ax_r.scatter([xa, xb + offset_r], [ya, yb], s=120, c='red', edgecolor='white', zorder=5)
            ax_r.set_title(f"A({mA['side']}) vs B({mB['side']}) Matches - Best Conf={best_conf:.3f}")
        else:
            ax_r.set_title(f"A({mA['side']}) vs B({mB['side']}) Matches - NO MATCHES FOUND")

        ax_r.axis('off')
        plt.tight_layout()
        plt.show()

# Print global best result summary
if global_best_data is not None:
    best_conf = global_best_conf
    A_side, B_side = global_best_data['A_side'], global_best_data['B_side']
    print(f"\n>>> BEST OVERALL MATCH: A (Mask {global_best_data['A_idx']}, {A_side}) vs B (Mask {global_best_data['B_idx']}, {B_side}) with confidence {best_conf:.3f} <<<\n")
else:
    print("\n>>> NO MATCHES FOUND ACROSS ANY PAIRS <<<")